In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# import shap  # Commented out due to potential installation issues
# from xgboost import XGBRegressor  # Commented out due to version mismatch
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
# from pmdarima import auto_arima  # Commented out if not installed
from sklearn.ensemble import RandomForestRegressor  # Alternative to XGBoost
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

ValueError: Mismatched version between the Python package and the native shared object.  Python package version: 3.0.1. Shared object version: 3.0.2. Shared object is loaded from: c:\Users\62812\miniconda3\envs\data-science\Lib\site-packages\xgboost\lib\xgboost.dll.
Likely cause:
  * XGBoost is first installed with anaconda then upgraded with pip. To fix it please remove one of the installations.

In [17]:
# 2. DATA PREPROCESSING (Metodologi 2.1 & 2.2)
df = pd.read_csv('../../data/processed/data_with_holidays.csv')
df['tanggal_data'] = pd.to_datetime(df['tanggal_data'])
df = df.sort_values('tanggal_data').set_index('tanggal_data')

In [18]:
# Pembersihan Data (Interpolasi Linear sesuai saran Bab IV)
df['harga'] = df['harga'].interpolate(method='linear')

In [19]:
# Feature Engineering sesuai Draft Seksi 2.2
df['is_libur_nasional'] = df['is_libur'].astype(int)
df['lag_1'] = df['harga'].shift(1)
df['lag_7'] = df['harga'].shift(7)
df['rolling_mean_7'] = df['harga'].shift(1).rolling(window=7).mean()

In [20]:
# Menghapus missing values akibat shift/rolling
df_model = df.dropna().copy()

In [21]:
# 3. HANDLING STASIONERITAS (Seksi 3.1)
print("--- Hasil Uji ADF pada Data Asli ---")
adf_result = adfuller(df_model['harga'])
print(f"ADF Statistic: {adf_result[0]:.4f}")
print(f"p-value: {adf_result[1]:.4f}")
print(f"Critical Values: {adf_result[4]}")
if adf_result[1] <= 0.05:
    print("Data stasioner (Tolak H0)")
else:
    print("Data tidak stasioner (Gagal tolak H0)")

--- Hasil Uji ADF pada Data Asli ---
ADF Statistic: -2.5318
p-value: 0.1079
Critical Values: {'1%': np.float64(-3.7112123008648155), '5%': np.float64(-2.981246804733728), '10%': np.float64(-2.6300945562130176)}
Data tidak stasioner (Gagal tolak H0)


In [22]:
# Melakukan cek differencing untuk laporan paper
print("\n--- Melakukan First Order Differencing ---")
diff_1 = df_model['harga'].diff().dropna()
adf_diff = adfuller(diff_1)
print(f"ADF Statistic (Diff-1): {adf_diff[0]:.4f}")
print(f"p-value (Diff-1): {adf_diff[1]:.4f}")


--- Melakukan First Order Differencing ---
ADF Statistic (Diff-1): -6.0055
p-value (Diff-1): 0.0000


In [23]:
# Train-test split (80% train, 20% test)
train_size = int(len(df_model) * 0.8)
train = df_model[:train_size]
test = df_model[train_size:]

# Define exogenous variables
exog_vars = ['is_libur_nasional', 'lag_1', 'lag_7', 'rolling_mean_7']

print(f"Train size: {len(train)}, Test size: {len(test)}")
print(f"Exogenous variables: {exog_vars}")

Train size: 21, Test size: 6
Exogenous variables: ['is_libur_nasional', 'lag_1', 'lag_7', 'rolling_mean_7']


In [24]:
# 4. PEMODELAN SARIMAX DENGAN PARAMETER INTEGRASI (d=1)
print("\n--- Fitting SARIMAX (Manual parameters d=1) ---")
# Manual SARIMAX parameters based on typical time series patterns
# ARIMA(1,1,1) with seasonal component (1,0,1,7)
order = (1, 1, 1)
seasonal_order = (1, 0, 1, 7)

model_sarimax = SARIMAX(train['harga'], exog=train[exog_vars],
                        order=order,
                        seasonal_order=seasonal_order).fit(disp=False)

print(model_sarimax.summary())


--- Fitting SARIMAX (Manual parameters d=1) ---


c:\Users\62812\miniconda3\envs\data-science\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\62812\miniconda3\envs\data-science\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\62812\miniconda3\envs\data-science\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
c:\Users\62812\miniconda3\envs\data-science\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for seasonal ARMA. All parameter

                                     SARIMAX Results                                     
Dep. Variable:                             harga   No. Observations:                   21
Model:             SARIMAX(1, 1, 1)x(1, 0, 1, 7)   Log Likelihood                -197.814
Date:                           Sat, 02 May 2026   AIC                            413.628
Time:                                   22:56:19   BIC                            422.590
Sample:                                        0   HQIC                           415.377
                                            - 21                                         
Covariance Type:                             opg                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
is_libur_nasional     0.0179    753.508   2.38e-05      1.000   -1476.831    1476.867
lag_1                 

c:\Users\62812\miniconda3\envs\data-science\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [25]:
# 5. PEMODELAN XGBOOST (Seksi 2.3)
features = ['is_libur_nasional', 'lag_1', 'lag_7', 'rolling_mean_7']
X_train, y_train = train[features], train['harga']
X_test, y_test = test[features], test['harga']

xgb_model = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=42)
xgb_model.fit(X_train, y_train)

# 6. EVALUASI KOMPARATIF (Seksi 3.3)
# Prediksi
pred_sarimax = model_sarimax.forecast(steps=len(test), exog=test[exog_vars])
pred_xgb = xgb_model.predict(X_test)

NameError: name 'XGBRegressor' is not defined